In [ ]:
from pathlib import Path
import math
import pandas as pd

file = Path("/kaggle/input/notebooks/urajkaushik/notebook191c8a3bcd/candidate_consensus.csv")
df = pd.read_csv(file)

transformer_col = "transformer_percentile"
forest_col = "isolation_forest_holdout_percentile"
paired = df.dropna(subset=[transformer_col, forest_col]).copy()

k = max(1, math.ceil(len(paired) * 0.05))
transformer_top = set(paired.nlargest(k, transformer_col)["source_id"])
forest_top = set(paired.nlargest(k, forest_col)["source_id"])

print("Sources scored by both models:", len(paired))
print("Spearman rank agreement:", round(
    paired[transformer_col].corr(paired[forest_col], method="spearman"), 3
))
print(f"Shared candidates in each model's top 5%: {len(transformer_top & forest_top)} of {k}")

paired.to_csv("/kaggle/working/day2_paired_model_scores.csv", index=False)
print("Saved paired scores: /kaggle/working/day2_paired_model_scores.csv")

In [ ]:
import math
import pandas as pd

paired = pd.read_csv("/kaggle/working/day2_paired_model_scores.csv")

transformer_col = "transformer_percentile"
forest_col = "isolation_forest_holdout_percentile"
k = math.ceil(len(paired) * 0.05)

transformer_top = set(paired.nlargest(k, transformer_col).index)
forest_top = set(paired.nlargest(k, forest_col).index)

paired["transformer_top5"] = paired.index.isin(transformer_top)
paired["isolation_forest_top5"] = paired.index.isin(forest_top)

paired["model_group"] = "neither_top5"
paired.loc[paired["transformer_top5"] & ~paired["isolation_forest_top5"], "model_group"] = "transformer_only"
paired.loc[paired["isolation_forest_top5"] & ~paired["transformer_top5"], "model_group"] = "isolation_forest_only"
paired.loc[paired["transformer_top5"] & paired["isolation_forest_top5"], "model_group"] = "both_top5"

review = paired[paired["model_group"] != "neither_top5"].copy()
review = review.sort_values("consensus_percentile", ascending=False)
review.to_csv("/kaggle/working/day2_top5_union_review.csv", index=False)

print("Saved top-candidate review:", len(review), "rows")
print(review["model_group"].value_counts())
display(review[[
    "source_id", "sector", "model_group",
    transformer_col, forest_col, "consensus_percentile",
    "evidence_level", "gaia_class_name", "tess_eb_catalog_match"
]])

In [ ]:
import pandas as pd

review = pd.read_csv("/kaggle/working/day2_top5_union_review.csv")

groups = {
    "both_top5": "consensus_percentile",
    "transformer_only": "transformer_percentile",
    "isolation_forest_only": "isolation_forest_holdout_percentile",
}

shortlist_parts = []
for group, rank_column in groups.items():
    part = (
        review[review["model_group"] == group]
        .sort_values(rank_column, ascending=False)
        .head(10)
        .copy()
    )
    part["priority_within_group"] = range(1, len(part) + 1)
    shortlist_parts.append(part)

shortlist = pd.concat(shortlist_parts, ignore_index=True)
output = "/kaggle/working/day3_candidate_vetting_shortlist.csv"
shortlist.to_csv(output, index=False)

print("Saved:", output)
print(shortlist["model_group"].value_counts())
display(shortlist[[
    "source_id", "sector", "model_group",
    "transformer_percentile", "isolation_forest_holdout_percentile",
    "evidence_level", "gaia_class_name", "tess_eb_catalog_match"
]])

In [ ]:
from pathlib import Path

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
print("Manifest found:", (base / "manifest.csv").exists())
print("Raw FITS count:", len(list((base / "raw").glob("*.fits"))))

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
raw_dir = base / "raw"
manifest = pd.read_csv(base / "manifest.csv")
shortlist = pd.read_csv("/kaggle/working/day3_candidate_vetting_shortlist.csv")

manifest["_source"] = pd.to_numeric(manifest["source_id"], errors="coerce")
manifest["_sector"] = pd.to_numeric(manifest["sector"], errors="coerce")

def find_fits(source_id, sector):
    rows = manifest[
        (manifest["_source"] == int(source_id))
        & (manifest["_sector"] == int(sector))
        & (manifest["status"] == "COMPLETE")
    ]
    if rows.empty:
        return None
    filename = str(rows.iloc[0]["product_filename"])
    path = raw_dir / Path(filename).name
    return path if path.exists() else None

def read_light_curve(path):
    with fits.open(path, memmap=True) as hdus:
        for hdu in hdus:
            data = getattr(hdu, "data", None)
            names = getattr(data, "names", None)
            if not names:
                continue
            columns = {name.upper(): name for name in names}
            flux_name = columns.get("PDCSAP_FLUX") or columns.get("SAP_FLUX")
            if "TIME" not in columns or flux_name is None:
                continue

            time = np.asarray(data[columns["TIME"]], dtype=float)
            flux = np.asarray(data[flux_name], dtype=float)
            good = np.isfinite(time) & np.isfinite(flux) & (flux > 0)

            if "QUALITY" in columns:
                quality_good = good & (np.asarray(data[columns["QUALITY"]]) == 0)
                if quality_good.sum() >= 32:
                    good = quality_good

            if good.sum() < 32:
                return None, None

            time, flux = time[good], flux[good]
            return time, flux / np.median(flux)

    return None, None

out_dir = Path("/kaggle/working/day3_lightcurve_plots")
out_dir.mkdir(parents=True, exist_ok=True)
missing = []

for start in range(0, len(shortlist), 10):
    page = shortlist.iloc[start:start + 10]
    fig, axes = plt.subplots(5, 2, figsize=(14, 16))
    axes = axes.ravel()

    for ax, (_, row) in zip(axes, page.iterrows()):
        path = find_fits(row["source_id"], row["sector"])
        if path is None:
            missing.append((row["source_id"], row["sector"]))
            ax.set_title(f"TIC {row['source_id']} — FITS not found")
            ax.axis("off")
            continue

        time, flux = read_light_curve(path)
        if time is None:
            ax.set_title(f"TIC {row['source_id']} — no usable flux")
            ax.axis("off")
            continue

        ax.plot(time, flux, ".", ms=1, alpha=0.45)
        ax.set_title(
            f"TIC {row['source_id']} | S{int(row['sector'])} | "
            f"{row['model_group']} | consensus {row['consensus_percentile']:.3f}",
            fontsize=8,
        )
        ax.set_xlabel("Time (BTJD)")
        ax.set_ylabel("Normalized flux")

    for ax in axes[len(page):]:
        ax.axis("off")

    fig.tight_layout()
    output = out_dir / f"day3_lightcurves_page{start // 10 + 1}.png"
    fig.savefig(output, dpi=150)
    plt.show()
    print("Saved:", output)

print("Candidates whose FITS files were not found:", missing)